# Từ bước tiền xử lý bước 2

## Bước này tạo files:

- text_feat.npy (xài sentence_transformer)

## ref:

- https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

In [1]:
import re
import os
import torch
import numpy as np
import pandas as pd

In [4]:
PATH = "../data/2023"

In [5]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.preprocessed.parquet"))

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   itemID           35997 non-null  int64  
 1   asin             35997 non-null  str    
 2   main_category    35997 non-null  str    
 3   title            35997 non-null  str    
 4   average_rating   35997 non-null  float64
 5   rating_number    35997 non-null  int64  
 6   features         35997 non-null  object 
 7   description      35997 non-null  object 
 8   price            17404 non-null  float64
 9   images           35997 non-null  object 
 10  videos           35997 non-null  object 
 11  store            35997 non-null  str    
 12  categories       35997 non-null  object 
 13  details          35997 non-null  object 
 14  bought_together  0 non-null      float64
 15  subtitle         0 non-null      float64
 16  author           0 non-null      object 
 17  combined_text    35997 

In [8]:
df.head(3)

,itemID,asin,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,bought_together,subtitle,author,combined_text,sentences
0,0,B086QM7FVT,Baby,"Skip Hop Toddler Step Stool, Double Up",4.6,2482,"[A big-kid boost to toddler independence, our ...","[A big-kid boost to toddler independence, our ...",21.99,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'My favorite step stool', 'url': 'h...",Skip Hop,"[Baby Products, Nursery, Furniture, Storage & ...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Skip Hop Toddler Step Stool, Double Up Skip Ho...","Skip Hop Toddler Step Stool, Double Up Skip Ho..."
1,1,B017IQZ9OK,Baby,"Boon Spring Countertop Drying Rack, Green (B11...",4.8,3885,[Drying rack: countertop drying rack holds ite...,[Boon Sprig countertop drying rack easily hold...,12.99,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Versatile and space saving UPDATE ...,Boon,"[Baby Products, Feeding, Bottle-Feeding, Bottl...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Boon Spring Countertop Drying Rack, Green (B11...","Boon Spring Countertop Drying Rack, Green (B11..."
2,2,B08FZJ3YHH,Baby,Toilet Seat Covers Disposable - 20 Pack - Wate...,4.8,7996,[✔️ NO MORE STRESS when you need to use a publ...,[],15.97,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Toilet Seat Covers Disposable: Fun...,Relyo,"[Baby Products, Potty Training, Seat Covers]","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,Toilet Seat Covers Disposable - 20 Pack - Wate...,Toilet Seat Covers Disposable - 20 Pack - Wate...


## Text Feature Extraction

In [ ]:
# %pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %pip install ipywidgets

In [ ]:
# should `pip install sentence_transformers` first
# should use gpu to speed up
# https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2, all-MiniLM-L12-v2, all-mpnet-base-v2 gpu (31s) (36s) (3p)

model_name = "all-MiniLM-L12-v2"

model = SentenceTransformer(model_name)

In [ ]:
# !pip install fastembed -q

In [4]:
# from fastembed import TextEmbedding

# # https://qdrant.github.io/fastembed/examples/Supported_Models/#supported-text-embedding-models
# model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
# print("Đang encode...")

# embeddings_generator = model.embed(sentences)
# sentence_embeddings = np.array(list(embeddings_generator))

# print(f"Shape: {sentence_embeddings.shape}")


# np.save(os.path.join('text_feat.npy'), sentence_embeddings)
# print('Đã lưu xong bằng FastEmbed!')

In [245]:
!nvidia-smi

Sat Mar 28 22:31:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   39C    P4             11W /   74W |    1368MiB /   6141MiB |     31%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [59]:
sentences = df["sentences"].tolist()

In [60]:
sentences[:3]

['Skip Hop Toddler Step Stool, Double Up Skip Hop Baby Baby Products Nursery Furniture Storage  &  Organization Step Stools A big-kid boost to toddler independence, our 2-in-1 step stool makes everything easier for kids to reach. Perfect at the bathroom sink and toilet for potty training, it’s also great as a kitchen helper and more Use the space-saving nesting stools together as a stepped design for an easier climb, or separately Interlocking design keeps stools securely attached when used together Featuring: Wide standing platforms, Non-slip bases  &  treads, Space-saving nesting design Size (inches): Large step stool: 13.25W x 8.5H x 10.25D; Small step stool: 10.375W x 6H x 10.25D A big-kid boost to toddler independence, our 2-in-1 step stool makes everything easier for kids to reach. Perfect at the bathroom sink and toilet for potty training, it’s also great as a kitchen helper and more. Use the space-saving nesting stools together as a stepped design for an easier climb, or separa

In [61]:
print("Bắt đầu Encode... Vui lòng đợi trong giây lát.")
with torch.no_grad():
    sentence_embeddings = model.encode(
        sentences, batch_size=64, show_progress_bar=True, convert_to_numpy=True
    )

print(f"Text encoded! Shape: {sentence_embeddings.shape}")

assert sentence_embeddings.shape[0] == df.shape[0]
save_path = os.path.join(PATH, "text_feat.npy")
np.save(save_path, sentence_embeddings)

print(f"Done! File đã được lưu tại: {save_path}")

Bắt đầu Encode... Vui lòng đợi trong giây lát.


Batches: 100%|██████████| 563/563 [12:35<00:00,  1.34s/it]


Text encoded! Shape: (35997, 384)
Done! File đã được lưu tại: ./data/2023\text_feat.npy


In [ ]:
# !pip install nvitop

In [ ]:
# !nvitop

In [62]:
sentence_embeddings[:10]

array([[-0.02511088,  0.050413  ,  0.04400188, ...,  0.04795245,
         0.07556733,  0.05827542],
       [-0.02860759,  0.0548236 , -0.00681664, ..., -0.03452795,
        -0.04217557,  0.05453426],
       [ 0.0636656 ,  0.01261443,  0.00849598, ...,  0.02520804,
         0.04014093,  0.06704497],
       ...,
       [ 0.02174732,  0.01410911, -0.00569856, ...,  0.05042551,
        -0.0428423 , -0.02519943],
       [ 0.03739659, -0.02329124,  0.00790992, ...,  0.02438011,
         0.04282311, -0.04832435],
       [-0.04577953, -0.00894414,  0.06457003, ...,  0.03238399,
        -0.00303201,  0.02178374]], shape=(10, 384), dtype=float32)